# MVA Track 2 · NB09
## Exposure–mechanism audit for the conditional proteostasis panel

**Question:** Are the mechanistic concentrations that motivate each candidate plausibly related to human exposure or demonstrated human target engagement?

This notebook does **not** infer efficacy in MVA. It quantifies exposure bridges and makes missing exposure data explicit.


In [ ]:
import pandas as pd, numpy as np, math, json
from pathlib import Path
OUT=Path("outputs"); OUT.mkdir(exist_ok=True)


## 1. Source-locked quantitative inputs

Values below are deliberately sparse and traceable.

- **Arimoclomol:** pediatric steady-state Cmax 523 ng/mL; plasma protein binding ~10%; MW 313.78 g/mol.
- **4-PBA / phenylbutyrate:** adult fasting Cmax 218 µg/mL after 5 g; phenylbutyrate MW 164.20 g/mol.
- **GGA / teprenone:** rather than forcing a serum concentration bridge, use direct human pharmacodynamic evidence: 150 mg/day for 2 weeks increased gastric mucosal HSP70.
- **Ambroxol:** high-dose 1.26 g/day produced a 156 ng/mL increase in CSF concentration; ambroxol MW 378.10 g/mol. Published GCase chaperone experiments commonly used 40–60 µM.

Important: concentrations from different models/tissues are *contextual comparators*, not interchangeable efficacy thresholds.


In [ ]:
rows=[
 {
  "drug":"arimoclomol","human_metric":"pediatric steady-state serum Cmax",
  "human_value":523,"human_unit":"ng/mL","mw_g_mol":313.78,"protein_binding_fraction":0.10,
  "human_total_uM":523/313.78,
  "human_free_uM":(523/313.78)*0.90,
  "mechanistic_low_uM":100,"mechanistic_high_uM":400,
  "bridge_type":"plasma PK vs nonclinical proteostasis concentrations",
  "source_human":"DailyMed MIPLYFFA, section 12.3",
  "source_mechanism":"EMA withdrawal assessment range cited in project review; verify exact page before final report"
 },
 {
  "drug":"sodium phenylbutyrate / 4-PBA","human_metric":"phenylbutyrate Cmax after 5 g fasting",
  "human_value":218,"human_unit":"ug/mL","mw_g_mol":164.20,"protein_binding_fraction":np.nan,
  "human_total_uM":218000/164.20,
  "human_free_uM":np.nan,
  "mechanistic_low_uM":5000,"mechanistic_high_uM":5000,
  "bridge_type":"plasma PK vs chemical-chaperone cell exemplar",
  "source_human":"DailyMed BUPHENYL",
  "source_mechanism":"PMID 26457178, 5 mM PBA"
 },
 {
  "drug":"geranylgeranylacetone / teprenone","human_metric":"direct human PD",
  "human_value":150,"human_unit":"mg/day","mw_g_mol":np.nan,"protein_binding_fraction":np.nan,
  "human_total_uM":np.nan,"human_free_uM":np.nan,
  "mechanistic_low_uM":np.nan,"mechanistic_high_uM":np.nan,
  "bridge_type":"human tissue target engagement",
  "source_human":"PMID 17684364: 150 mg/day x 2 weeks increased gastric mucosal HSP70",
  "source_mechanism":"same human PD study"
 },
 {
  "drug":"ambroxol","human_metric":"increase in CSF concentration at 1.26 g/day",
  "human_value":156,"human_unit":"ng/mL","mw_g_mol":378.10,"protein_binding_fraction":np.nan,
  "human_total_uM":156/378.10,
  "human_free_uM":np.nan,
  "mechanistic_low_uM":40,"mechanistic_high_uM":60,
  "bridge_type":"CSF PK vs GCase chaperone cell concentrations",
  "source_human":"PMID 31930374",
  "source_mechanism":"PMID 19578116 (40 uM); PMID 27539639 (60 uM)"
 }
]
D=pd.DataFrame(rows)
D


## 2. Nominal concentration-gap calculation

A ratio is calculated only when both a human concentration and a mechanistic concentration are available.  
It is **not** interpreted as a dose conversion.


In [ ]:
D["gap_low_x"]=D["mechanistic_low_uM"]/D["human_total_uM"]
D["gap_high_x"]=D["mechanistic_high_uM"]/D["human_total_uM"]

def call(r):
    if r["drug"].startswith("geranylgeranyl"):
        return "HUMAN_PD_TARGET_ENGAGEMENT_LOCAL_TISSUE"
    lo=r["gap_low_x"]
    if pd.isna(lo): return "INSUFFICIENT_EXPOSURE_DATA"
    if lo<=5: return "NOMINAL_EXPOSURE_OVERLAP"
    if lo<=20: return "MODERATE_NOMINAL_GAP"
    return "LARGE_NOMINAL_GAP"

D["exposure_gate"]=D.apply(call,axis=1)
cols=["drug","human_metric","human_total_uM","human_free_uM","mechanistic_low_uM",
      "mechanistic_high_uM","gap_low_x","gap_high_x","exposure_gate","bridge_type"]
display(D[cols])
D.to_csv(OUT/"NB09_EXPOSURE_MECHANISM_AUDIT.csv",index=False)


## 3. Interpretation

### Arimoclomol
The human pediatric Cmax is approximately **1.67 µM total** and, using 10% protein binding, approximately **1.50 µM nominally unbound**.  
If the relevant mechanistic literature indeed requires 100–400 µM, the nominal gap is large. This is a reason to **downgrade confidence and require an exposure-response experiment**, not proof of inefficacy in MVA.

### 4-PBA
A 5 g fasting dose gives phenylbutyrate Cmax of roughly **1.33 mM**. A 5 mM chemical-chaperone exemplar is therefore only several-fold above total Cmax, making the exposure bridge materially less implausible than for arimoclomol. Protein binding, intracellular exposure and chronic dosing still need explicit treatment.

### GGA
The strongest bridge is not a guessed plasma concentration. A clinical dose (150 mg/day) increased HSP70 in human gastric mucosa. This is **direct human pharmacodynamic support for the HSP-induction mechanism**, but it does not establish target engagement in dividing RPE1 cells or BUBR1 rescue.

### Ambroxol
High-dose human treatment demonstrated CSF penetration and altered GCase protein biology, but nominal CSF concentration (~0.41 µM) is much lower than classic 40–60 µM GCase chaperone experiments. Because the target is GCase rather than BUBR1, this remains a **secondary mechanistic comparator**, not a direct exposure argument for BUBR1.


## 4. Preclinical rescue exposure rule

A candidate can advance from "mechanistically interesting" to "pharmacologically testable" only if the experiment includes:

1. an exposure range spanning **clinically observed concentration** where known;
2. higher concentrations only as clearly labelled mechanistic probes;
3. intracellular/target engagement when measurable;
4. chromosome missegregation per completed division;
5. matched proliferation/mitotic output and viability;
6. no rescue credit for cytostasis or selective killing.

For arimoclomol specifically, the first experiment should include a low-µM clinically anchored range before any 100+ µM mechanistic range.


In [ ]:
handoff={
 "schema":"MVA-NB09-EXPOSURE-MECHANISM-1.0.0",
 "ranking_used":False,
 "purpose":"exposure_plausibility_gate",
 "primary_output":"NB09_EXPOSURE_MECHANISM_AUDIT.csv",
 "key_conclusion":"Approved/marketed status is not sufficient; exposure bridge differs materially across candidates.",
 "arimoclomol_status":"MAJOR_NOMINAL_EXPOSURE_GAP_REQUIRES_LOW_uM_EXPERIMENT",
 "pba_status":"NOMINAL_TOTAL_CMAX_WITHIN_SEVERAL_FOLD_OF_5mM_EXEMPLAR",
 "gga_status":"DIRECT_HUMAN_LOCAL_TISSUE_HSP70_PD_SUPPORT",
 "ambroxol_status":"HUMAN_CSF_TARGET_ENGAGEMENT_BUT_LARGE_NOMINAL_GAP_TO_CLASSIC_CELL_CONCENTRATIONS"
}
(OUT/"NB09_HANDOFF.json").write_text(json.dumps(handoff,indent=2))
print(json.dumps(handoff,indent=2))
